# Agentic datasheet Q&A (Microsoft Agent Framework)

`answer_question(question) -> AgentResult` answers a question about the ingested datasheets (see `pdf_ingestion_qdrant.ipynb`).

Unlike fixed RAG (one search, then answer), the agent **decides** how to get the evidence it needs:

```
answer_question(question)
│
├─ Agent  (Microsoft Agent Framework · OpenAI Responses API · gpt-5.5)
│    plans → searches (targeted by content type / part) → expands context → looks at page images → calculates
│    tools: list_documents · search_datasheets · get_table · get_page · get_neighbor_chunks
│           view_page_image · calculate
│    output: DatasheetAnswer (structured: answer, conditions, basis, citations, confidence, found_in_datasheet)
│
├─ Verifier (separate agent, no tools): checks every claim against the evidence the agent actually read
│    └─ if not supported → one revision round in the same session (the agent can search again) → re-verify
│
└─ AgentResult: answer + verdict + evidence ledger + tool calls + token usage + latency + Phoenix trace id
```

Every run is traced to Phoenix (project **datasheet-agent**, http://localhost:6006): agent → each LLM step → each tool call → retrieval documents → verifier.

**Prerequisites:** `docker compose up -d` (Qdrant + Phoenix), the collection ingested, and `uv sync`, then select the `.venv` kernel.

## 1. Setup

In [1]:
import ast, asyncio, json, math, operator, time
import contextvars
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Annotated, Literal

from pydantic import BaseModel, Field
from IPython.display import Markdown, display

from datasheet_rag import config
from datasheet_rag.tracing import setup_tracing, tracer, flush
from datasheet_rag import retrieval

from agent_framework import Agent, Content, FunctionInvocationConfiguration, add_usage_details, tool
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

AGENT_MODEL     = "gpt-5.5"
AGENT_EFFORT     = "medium"     # reasoning effort for the agent (planning + tool use)
VERIFIER_MODEL   = "gpt-5.5"
VERIFIER_EFFORT  = "low"
ENABLE_VERIFIER  = True
MAX_REVISIONS    = 1            # revision rounds after a failed verification
MAX_ITERATIONS   = 15           # max model↔tool round-trips per agent run
PHOENIX_PROJECT  = "datasheet-agent"

RUN_DEMO = globals().get("RUN_DEMO", True)   # the eval notebook sets RUN_DEMO=False before `%run -i` of this notebook

tracer_provider = setup_tracing(PHOENIX_PROJECT, agent_framework=True)
print("Datasheets:", [d["title"] for d in retrieval.list_documents()])

Tracing to http://localhost:6006  project='datasheet-agent'


Datasheets: ['5V, 10A Synchronous Step-Down Silent Switcher 2 in 3mm × 3mm LQFN']


## 2. Evidence ledger

Each tool call records what it returned in a per-question ledger, held in a `ContextVar` so concurrent questions don't mix. The ledger is used for:
- the **verifier**, which checks the answer only against what the agent actually read;
- the **result**, so callers and the TruLens evals can see the retrieved contexts.

In [2]:
_LEDGER: contextvars.ContextVar[list | None] = contextvars.ContextVar("evidence_ledger", default=None)
TOOL_TEXT_LIMIT = 12_000      # chars of each tool result passed to the model

def _record(tool_name: str, args: dict, content: str) -> str:
    content = content if len(content) <= TOOL_TEXT_LIMIT else content[:TOOL_TEXT_LIMIT] + "\n…(truncated)"
    if (ledger := _LEDGER.get()) is not None:
        ledger.append({"tool": tool_name, "args": args, "content": content})
    return content

def _compact_hit(h: dict) -> dict:
    ref = {k: h[k] for k in ("doc_id", "chunk_index", "content_type", "page_start", "section", "table_id",
                             "table_title", "figure_title", "pin_name", "score") if h.get(k) not in (None, "")}
    return {"ref": ref, "text": h["text"]}

## 3. Tools

The descriptions are written for the model: they say *when* to use each tool, not just what it does. The parameters are typed, so Agent Framework generates exact JSON schemas from them.

In [3]:
ContentType = Literal["text", "table", "figure", "pin", "page"]

@tool(description="List the datasheets in the knowledge base: doc_id, title, part numbers, device category, summary, page count. "
                  "Call first when you need doc_id values or to check which devices are covered.")
def list_documents() -> str:
    docs = [{k: d.get(k) for k in ("doc_id", "source", "title", "part_numbers", "device_category", "doc_summary", "num_pages")}
            for d in retrieval.list_documents()]
    return _record("list_documents", {}, json.dumps(docs, ensure_ascii=False))

@tool(description="Hybrid semantic + keyword search over the datasheets. Returns chunks with a ref (doc_id, chunk_index, page, "
                  "section, table_id/figure_title/pin_name) and text. Use datasheet vocabulary (parameter, pin, symbol names). "
                  "Target with content_types: 'table' = specifications (electrical characteristics, abs max, ordering), "
                  "'figure' = typical performance graphs / application circuits / block diagram, 'pin' = pin functions, "
                  "'text' = applications info, equations, operation. Run several focused searches rather than one broad one.")
def search_datasheets(
    query: Annotated[str, Field(description="Focused search query")],
    content_types: Annotated[list[ContentType] | None, Field(description="Restrict to these chunk types")] = None,
    part_number: Annotated[str | None, Field(description="Restrict to datasheets covering this base part, e.g. 'LTC3310S'")] = None,
    top_k: Annotated[int, Field(ge=1, le=12, description="Number of results")] = 6,
) -> str:
    hits = retrieval.search_datasheets(query, top_k=top_k, content_types=content_types, part_number=part_number)
    return _record("search_datasheets", {"query": query, "content_types": content_types, "part_number": part_number},
                   json.dumps([_compact_hit(h) for h in hits], ensure_ascii=False))

@tool(description="Return a COMPLETE table (all row groups, column header, table note with test conditions) by doc_id + table_id "
                  "from a search hit. Use when a table hit may be missing rows or you need the conditions/notes.")
def get_table(doc_id: str, table_id: str) -> str:
    return _record("get_table", {"doc_id": doc_id, "table_id": table_id},
                   json.dumps(retrieval.get_table(doc_id, table_id), ensure_ascii=False) or "table not found")

@tool(description="Return the full parsed Markdown of one page (1-based). Use for surrounding context, equations, footnotes "
                  "(e.g. 'Note 2'), or everything on a page that a hit came from.")
def get_page(doc_id: str, page: Annotated[int, Field(ge=1)]) -> str:
    return _record("get_page", {"doc_id": doc_id, "page": page},
                   json.dumps(retrieval.get_page(doc_id, page), ensure_ascii=False) or "page not found")

@tool(description="Return the chunks immediately before/after a hit (reading order) to continue text that was cut off.")
def get_neighbor_chunks(doc_id: str, chunk_index: int,
                        window: Annotated[int, Field(ge=1, le=3)] = 1) -> str:
    hits = retrieval.get_neighbor_chunks(doc_id, chunk_index, window)
    return _record("get_neighbor_chunks", {"doc_id": doc_id, "chunk_index": chunk_index, "window": window},
                   json.dumps([_compact_hit(h) for h in hits], ensure_ascii=False))

def _image_result(result) -> list[Content]:
    meta, png = result
    return [Content.from_text(json.dumps(meta)), Content.from_data(data=png, media_type="image/png")]

@tool(description="Look at the ORIGINAL rendered PDF page as an image. Use to read a graph value precisely, check a schematic, "
                  "or resolve an ambiguous/garbled table. Text from the page is available via get_page; use this for visuals.",
      result_parser=_image_result)
def view_page_image(doc_id: str, page: Annotated[int, Field(ge=1)]):
    png = retrieval.render_page_image(doc_id, page, dpi=150)
    _record("view_page_image", {"doc_id": doc_id, "page": page},
            f"[viewed original page image: page {page}. Graph readings taken from it are approximate; the parsed "
            f"description of the same figure is the textual evidence.]")
    return {"doc_id": doc_id, "page": page, "note": "rendered page image follows"}, png

# ---- safe calculator: arithmetic + math functions only, no eval() ----
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod, ast.FloorDiv: operator.floordiv,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
_FUNCS = {n: getattr(math, n) for n in ("sqrt", "log", "log10", "log2", "exp", "sin", "cos", "tan", "atan", "ceil", "floor")}
_FUNCS |= {"abs": abs, "min": min, "max": max, "round": round}
_CONSTS = {"pi": math.pi, "e": math.e}

def _safe_eval(node):
    match node:
        case ast.Expression(body=b):                            return _safe_eval(b)
        case ast.Constant(value=v) if isinstance(v, (int, float)): return v
        case ast.Name(id=n) if n in _CONSTS:                    return _CONSTS[n]
        case ast.UnaryOp(op=op, operand=x) if type(op) in _OPS: return _OPS[type(op)](_safe_eval(x))
        case ast.BinOp(left=l, op=op, right=r) if type(op) in _OPS:
            a, b = _safe_eval(l), _safe_eval(r)
            if isinstance(op, ast.Pow) and abs(b) > 100:
                raise ValueError("exponent too large")
            return _OPS[type(op)](a, b)
        case ast.Call(func=ast.Name(id=f), args=args, keywords=[]) if f in _FUNCS:
            return _FUNCS[f](*[_safe_eval(a) for a in args])
    raise ValueError(f"unsupported expression element: {ast.dump(node)[:80]}")

@tool(description="Evaluate an arithmetic expression exactly (+ - * / ** % sqrt log log10 exp sin cos tan atan ceil floor "
                  "abs min max round pi e). Use for EVERY calculation (equations from the datasheet, unit conversions). "
                  "Pass plain numbers in consistent units, e.g. '41.2 * (1/1.0) ** 1.08' or '1.2/(3*1.0) * (1 - 1.2/5)'.")
def calculate(expression: Annotated[str, Field(description="Arithmetic expression")],
              purpose: Annotated[str, Field(description="What is being computed, with units")] = "") -> str:
    try:
        value = _safe_eval(ast.parse(expression, mode="eval"))
        out = json.dumps({"expression": expression, "purpose": purpose, "result": value})
    except Exception as e:
        out = json.dumps({"expression": expression, "error": str(e)})
    return _record("calculate", {"expression": expression, "purpose": purpose}, out)

TOOLS = [list_documents, search_datasheets, get_table, get_page, get_neighbor_chunks, view_page_image, calculate]
assert abs(_safe_eval(ast.parse("1.2/(3*1.0) * (1 - 1.2/5)", mode="eval")) - 0.304) < 1e-9
print("Tools:", [t.name for t in TOOLS])

Tools: ['list_documents', 'search_datasheets', 'get_table', 'get_page', 'get_neighbor_chunks', 'view_page_image', 'calculate']


## 4. Structured outputs

In [4]:
class Citation(BaseModel):
    source: str = Field(description="PDF file name")
    page: int
    reference: str = Field(description="What on the page supports the claim, e.g. 'Electrical Characteristics: VIN Undervoltage Lockout'")

class DatasheetAnswer(BaseModel):
    answer: str = Field(description="Markdown answer for an engineer: direct answer first, then supporting detail, inline [p.N] citations")
    found_in_datasheet: bool = Field(description="False if the datasheets do not contain the requested information")
    conditions: str = Field(description="Test conditions / assumptions the values apply to (empty if not applicable)")
    basis: list[Literal["guaranteed spec", "typical spec", "graph reading", "calculation", "descriptive text",
                        "application circuit", "not found"]] = Field(description="Kinds of evidence the answer rests on")
    citations: list[Citation]
    confidence: Literal["high", "medium", "low"]

class Verdict(BaseModel):
    verdict: Literal["supported", "partially_supported", "unsupported"]
    issues: list[str] = Field(description="Specific material problems: unsupported/contradicted claims, wrong or missing "
                                          "conditions, wrong device variant, arithmetic errors, missing citations")
    feedback: str = Field(description="Concrete instructions to fix the answer (what to look up / correct); empty if supported")

## 5. Agent instructions

The system prompt sets out the working method (plan → retrieve → expand → compute → stop) and the domain rules a datasheet engineer follows:
- identify the exact device variant;
- give guaranteed limits vs typical values vs graph readings;
- always state test conditions;
- absolute maximum ratings are not operating conditions;
- no outside knowledge for specs.

In [5]:
AGENT_INSTRUCTIONS = """You are a senior applications engineer answering questions about semiconductor devices using ONLY the datasheets in the knowledge base, which you access through your tools.

How to work:
1. Plan. Work out which facts you need. Split multi-part questions into sub-questions. Identify the exact device variant (e.g. LTC3310S vs LTC3310, '-1' fixed-output versions, temperature grades). If the question does not specify a variant and the answer differs between variants, answer for each.
2. Retrieve. Call search_datasheets with focused queries in datasheet vocabulary (parameter names, pin names, symbols). Use content_types to target: 'table' for specifications, 'figure' for typical-performance graphs and application circuits, 'pin' for pin functions, 'text' for applications information and equations. Run several searches when needed; independent searches can be issued in parallel.
3. Expand. A search hit is a fragment: use get_table when a table might be missing rows or you need its test conditions, get_page or get_neighbor_chunks for surrounding text, equations and footnotes, and view_page_image to read a graph or schematic precisely or when parsed text looks ambiguous.
4. Compute. Use calculate for every arithmetic step. State the formula, the inputs and where each input comes from.
5. Stop when every part of the answer has direct evidence. If targeted searches show the datasheets do not contain the information, say so plainly and set found_in_datasheet=false. Never use outside knowledge for device specifications.

Answer rules:
- Every number comes from the tools, with units and the conditions it applies to (e.g. TA = 25°C, VIN = 3.3V, or 'over the full operating temperature range').
- Distinguish guaranteed limits (MIN/MAX), typical values (TYP) and approximate graph readings, and say which one you are giving. For a specification question prefer guaranteed limits and add the typical value when useful.
- Absolute maximum ratings are stress limits, not recommended operating conditions; say so when relevant.
- Cite pages inline like [p.3] and list every supporting source in citations.
- Be concise and engineer-friendly: direct answer first, then the supporting detail."""

VERIFIER_INSTRUCTIONS = """You are a strict technical reviewer of answers about semiconductor datasheets. You receive a question, a draft answer (JSON) and the EVIDENCE: every tool result the assistant actually read.

Check each material claim of the draft (numbers, units, MIN/TYP/MAX type, test conditions, device variant applicability, pin behaviour, equations and arithmetic, 'not found' statements) against the evidence:
- A claim must be directly supported by the evidence. Outside knowledge does not count.
- Arithmetic must follow from cited inputs (re-check it).
- Approximate graph readings taken from a page image are acceptable if consistent with the textual figure description in the evidence; flag them only if contradicted.
- A 'not found' answer is wrong if the evidence actually contains the answer.
- Flag missing or wrong conditions, wrong variant, mislabelled typical-vs-guaranteed values, and claims without citations.
Ignore style. verdict = 'supported' only if there are no material issues."""

## 6. `answer_question`

The flow for each question:
1. A root span `answer_question` is opened. The trace id is returned so you can find the trace in Phoenix.
2. The agent runs in a fresh `AgentSession`, with the iteration cap, tool errors returned to the model, and parallel tool calls allowed.
3. The verifier checks the structured answer against the evidence ledger.
4. If the answer isn't supported, the feedback goes back into the **same session** and the agent searches again as needed. The revised answer is then verified again.

In [6]:
@dataclass
class AgentResult:
    question: str
    answer: DatasheetAnswer
    verdict: Verdict | None
    revisions: int
    evidence: list[dict]
    tool_calls: list[dict]
    usage: dict
    latency_s: float
    trace_id: str
    draft_answer: DatasheetAnswer | None = None   # answer before revision, if any

    @property
    def contexts(self) -> list[str]:
        """Individual retrieved texts the agent read (chunks, tables, pages), de-duplicated in retrieval order.
        Used by the evals (context relevance / groundedness). Calculator and image placeholders are excluded."""
        out = []
        for e in self.evidence:
            if e["tool"] in ("calculate", "view_page_image", "list_documents"):
                continue
            try:
                data = json.loads(e["content"])
            except ValueError:                       # truncated / non-JSON result: keep as one block
                out.append(e["content"]); continue
            for item in (data if isinstance(data, list) else [data]):
                text = (item.get("text") or item.get("markdown")) if isinstance(item, dict) else None
                out.append(text or json.dumps(item, ensure_ascii=False))
        return list(dict.fromkeys(out))

def _make_client(http: AsyncOpenAI) -> OpenAIChatClient:
    # one client per run: Agent Framework clients must not be shared across event loops
    return OpenAIChatClient(
        model=AGENT_MODEL, async_client=http,
        function_invocation_configuration=FunctionInvocationConfiguration(
            max_iterations=MAX_ITERATIONS, max_consecutive_errors_per_request=3,
            include_detailed_errors=True, allow_concurrent_invocation=True),
    )

def _evidence_text(ledger: list[dict], limit: int = 120_000) -> str:
    parts, total = [], 0
    for i, e in enumerate(ledger, 1):
        block = f"### [{i}] {e['tool']}({json.dumps(e['args'], ensure_ascii=False)})\n{e['content']}\n"
        if total + len(block) > limit:
            parts.append("…(further evidence truncated)"); break
        parts.append(block); total += len(block)
    return "\n".join(parts) or "(no evidence retrieved)"

async def _verify(http: AsyncOpenAI, question: str, answer: DatasheetAnswer, ledger: list[dict]) -> tuple[Verdict, dict]:
    verifier = Agent(OpenAIChatClient(model=VERIFIER_MODEL, async_client=http), name="verifier", instructions=VERIFIER_INSTRUCTIONS,
                     default_options={"reasoning": {"effort": VERIFIER_EFFORT}, "response_format": Verdict})
    resp = await verifier.run(
        f"QUESTION:\n{question}\n\nDRAFT ANSWER (JSON):\n{answer.model_dump_json(indent=1)}\n\nEVIDENCE:\n{_evidence_text(ledger)}")
    return resp.value, resp.usage_details or {}

async def answer_question_async(question: str) -> AgentResult:
    t0, ledger, usage = time.perf_counter(), [], {}
    token = _LEDGER.set(ledger)
    http = AsyncOpenAI(max_retries=5, timeout=300)    # closed in `finally` while the event loop is still alive
    try:
        with tracer.start_as_current_span("answer_question", openinference_span_kind="chain") as span:
            span.set_input(question)
            trace_id = format(span.get_span_context().trace_id, "032x")

            agent = Agent(_make_client(http), name="datasheet_agent", instructions=AGENT_INSTRUCTIONS, tools=TOOLS,
                          default_options={"reasoning": {"effort": AGENT_EFFORT}, "response_format": DatasheetAnswer})
            session = agent.create_session()
            resp = await agent.run(question, session=session)
            usage = add_usage_details(usage, resp.usage_details)
            answer, draft, verdict, revisions = resp.value, None, None, 0

            if ENABLE_VERIFIER:
                verdict, u = await _verify(http, question, answer, ledger)
                usage = add_usage_details(usage, u)
                while verdict.verdict != "supported" and revisions < MAX_REVISIONS:
                    revisions += 1
                    draft = draft or answer
                    resp = await agent.run(
                        "A technical reviewer checked your answer against the evidence and found problems:\n- "
                        + "\n- ".join(verdict.issues) + f"\n\nReviewer guidance: {verdict.feedback}\n\n"
                        "Investigate with your tools as needed and return a corrected answer. "
                        "Remove any claim you cannot support with evidence.", session=session)
                    usage = add_usage_details(usage, resp.usage_details)
                    answer = resp.value
                    verdict, u = await _verify(http, question, answer, ledger)
                    usage = add_usage_details(usage, u)

            span.set_attributes({"agent.revisions": revisions, "agent.tool_calls": len(ledger),
                                 "agent.verdict": verdict.verdict if verdict else "not_verified",
                                 "agent.found_in_datasheet": answer.found_in_datasheet,
                                 "agent.confidence": answer.confidence})
            span.set_output(answer.model_dump_json(), mime_type="application/json")
    finally:
        _LEDGER.reset(token)
        await http.close()

    return AgentResult(question=question, answer=answer, verdict=verdict, revisions=revisions, evidence=ledger,
                       tool_calls=[{"tool": e["tool"], **e["args"]} for e in ledger], usage=dict(usage),
                       latency_s=round(time.perf_counter() - t0, 1), trace_id=trace_id, draft_answer=draft)

def answer_question(question: str) -> AgentResult:
    """Synchronous entry point (works in scripts and inside Jupyter's already-running event loop)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        result = asyncio.run(answer_question_async(question))
    else:
        with ThreadPoolExecutor(1) as pool:
            result = pool.submit(asyncio.run, answer_question_async(question)).result()
    flush()
    return result

## 7. Display helper

In [7]:
def show(result: AgentResult, details: bool = True) -> None:
    a, v = result.answer, result.verdict
    badge = {"supported": "✅ verified", "partially_supported": "⚠️ partially supported",
             "unsupported": "❌ not supported"}.get(v.verdict if v else "", "– not verified")
    md_ = [f"### Q: {result.question}", a.answer, ""]
    if a.conditions:
        md_.append(f"**Conditions:** {a.conditions}")
    md_.append(f"**Basis:** {', '.join(a.basis)} · **Confidence:** {a.confidence} · **Verifier:** {badge}"
               + (f" (after {result.revisions} revision)" if result.revisions else ""))
    md_.append("**Citations:** " + "; ".join(f"{c.source} p.{c.page}: {c.reference}" for c in a.citations))
    if v and v.issues:
        md_.append("**Open verifier issues:** " + "; ".join(v.issues))
    if details:
        steps = " → ".join(f"`{t['tool']}`" for t in result.tool_calls)
        tok = result.usage.get("total_token_count", "?")
        md_.append(f"<sub>{len(result.tool_calls)} tool calls: {steps}<br>{result.latency_s}s · {tok} tokens · "
                   f"Phoenix trace `{result.trace_id}`</sub>")
    display(Markdown("\n\n".join(md_)))

## 8. Try it

The questions below cover the different behaviours: a spec lookup with variant awareness, a multi-step design question with calculation, a graph reading compared against the spec, and a question the datasheet can't answer. The full evaluation is in `trulens_evals.ipynb`.

In [8]:
if RUN_DEMO:
    show(answer_question("What are the minimum and maximum VIN undervoltage lockout thresholds, and how much hysteresis is there?"))

### Q: What are the minimum and maximum VIN undervoltage lockout thresholds, and how much hysteresis is there?

For the LTC3310S/LTC3310S-1/LTC3310 family, the **VIN undervoltage lockout threshold is specified for VIN rising** as:

- **Minimum:** **2.0 V**
- **Typical:** **2.1 V**
- **Maximum:** **2.2 V**
- **Hysteresis:** **150 mV typical**; no min/max hysteresis limit is listed [p.3].

Using the typical values, the approximate **VIN falling UVLO threshold** is **2.1 V − 0.150 V = 1.95 V**. This falling value is calculated from typical specs only, not a guaranteed limit.



**Conditions:** Electrical Characteristics table conditions: VIN = 3.3 V, VEN = VIN, MODE/SYNC = 0 V, unless otherwise noted. The ● specs apply over the specified operating temperature range; otherwise specs are at TA = 25°C. VIN UVLO rising threshold is marked as applying over temperature; hysteresis is typical only.

**Basis:** guaranteed spec, typical spec, calculation · **Confidence:** high · **Verifier:** ✅ verified

**Citations:** ltc3310s-3310s-1-3310-3310-1.pdf p.3: Electrical Characteristics table: Input Supply rows for VIN Undervoltage Lockout and VIN Undervoltage Lockout Hysteresis

<sub>5 tool calls: `list_documents` → `search_datasheets` → `search_datasheets` → `get_table` → `calculate`<br>40.3s · 40904 tokens · Phoenix trace `d56027ab3005494f1b21aaa60ce37b8c`</sub>

In [9]:
if RUN_DEMO:
    show(answer_question("I want to run the LTC3310S at 1 MHz from a 5 V input with a 1.2 V output. "
                         "What RT resistor should I use, and what minimum inductor value does the datasheet's guideline give?"))

### Q: I want to run the LTC3310S at 1 MHz from a 5 V input with a 1.2 V output. What RT resistor should I use, and what minimum inductor value does the datasheet's guideline give?

Use **RT = 549 kΩ** for **1 MHz**; that value is listed directly in the LTC3310S “SW Frequency vs RT Value” table [p.14].

For the inductor guideline, with **VIN(MAX) = 5 V**, **VOUT = 1.2 V**, and **fSW = 1 MHz**:

- VOUT/VIN(MAX) = 1.2/5 = **0.24**, so use the datasheet’s equation for VOUT/VIN(MAX) ≤ 0.5 [p.15].
- L ≥ VOUT/(3 A · fSW) · (1 − VOUT/VIN(MAX))
- L ≥ 1.2/(3·1) · (1 − 1.2/5) = **0.304 µH**

So the datasheet guideline gives **L ≥ 0.304 µH**; in practice, choose the next suitable standard value, e.g. **0.33 µH**, with adequate RMS and saturation current ratings [p.15].

Note: the RT equation on the same page gives RT = 568·fSW^−1.08 = **568 kΩ** at 1 MHz, but since 1 MHz is explicitly tabulated, I would use the table value **549 kΩ** [p.14].



**Conditions:** LTC3310S, RT resistor from RT pin to ground for fSW = 1 MHz. Inductor calculation assumes VIN(MAX) = 5 V, VOUT = 1.2 V, fSW = 1 MHz; fSW in MHz and L in µH per datasheet equation.

**Basis:** descriptive text, calculation · **Confidence:** high · **Verifier:** ✅ verified

**Citations:** ltc3310s-3310s-1-3310-3310-1.pdf p.14: Applications Information: Setting the Switching Frequency; Table 1. SW Frequency vs RT Value; RT equation; ltc3310s-3310s-1-3310-3310-1.pdf p.15: Applications Information: Inductor Selection and Maximum Output Current; equations for first-choice inductor value

<sub>6 tool calls: `list_documents` → `search_datasheets` → `search_datasheets` → `calculate` → `calculate` → `calculate`<br>40.6s · 36421 tokens · Phoenix trace `c59b6cad36c56340ff3e59b6e7f87535`</sub>

In [10]:
if RUN_DEMO:
    show(answer_question("What is the PMOS (top switch) on-resistance at a 5 V input, and how does that compare to the value in the electrical characteristics table?"))

### Q: What is the PMOS (top switch) on-resistance at a 5 V input, and how does that compare to the value in the electrical characteristics table?

At **VIN = 5 V**, the PMOS/top-switch on-resistance is approximately **12.5 mΩ typical** from the “Switch On-Resistance” typical-performance graph [p.6].

The Electrical Characteristics table lists **Top Switch ON-Resistance = 16 mΩ typical** under the table’s default conditions: **VIN = 3.3 V, TA = 25°C, VEN = VIN, MODE/SYNC = 0 V unless otherwise noted** [p.4]. So the 5 V graph value is about **3.5 mΩ lower**, i.e. about **22% lower** than the table value; equivalently, the table value is about **1.28×** the 5 V graph reading.

Note that this is a comparison of a **typical graph reading at 5 V input** versus a **typical electrical-characteristics table value at 3.3 V input**, not a guaranteed limit.



**Conditions:** Graph value: Typical Performance Characteristics, TA = 25°C, PMOS curve at VIN = 5 V. Table value: Electrical Characteristics default conditions, VIN = 3.3 V, VEN = VIN, MODE/SYNC = 0 V, TA = 25°C unless otherwise noted.

**Basis:** typical spec, graph reading, calculation · **Confidence:** high · **Verifier:** ✅ verified

**Citations:** ltc3310s-3310s-1-3310-3310-1.pdf p.6: Typical Performance Characteristics: Switch On-Resistance graph, PMOS RDS(ON) vs input voltage; ltc3310s-3310s-1-3310-3310-1.pdf p.4: Electrical Characteristics: Top Switch ON-Resistance = 16 mΩ typical; table note with default conditions

<sub>8 tool calls: `list_documents` → `search_datasheets` → `search_datasheets` → `view_page_image` → `get_table` → `calculate` → `calculate` → `calculate`<br>42.7s · 54473 tokens · Phoenix trace `646e04fb567a93a63228f85c0e18d8fb`</sub>

In [11]:
if RUN_DEMO:
    show(answer_question("What is the unit price of the LTC3310S in 1k quantities?"))

### Q: What is the unit price of the LTC3310S in 1k quantities?

The datasheet in the knowledge base does **not** provide a 1k-quantity unit price for the LTC3310S. It only lists orderable LTC3310S part numbers/package options and temperature ranges in the Ordering Information table—there is no price or 1k-quantity pricing column [p.2]. For specific ordering information, the datasheet directs users to contact Analog Devices/local representative for some variants [p.3].



**Basis:** not found, descriptive text · **Confidence:** high · **Verifier:** ✅ verified

**Citations:** ltc3310s-3310s-1-3310-3310-1.pdf p.2: ORDER INFORMATION table lists LTC3310S orderable part numbers, package, marking, and temperature range but no pricing column; ltc3310s-3310s-1-3310-3310-1.pdf p.3: ORDER INFORMATION notes: contact representative/factory for specific product ordering information

<sub>4 tool calls: `list_documents` → `search_datasheets` → `get_table` → `get_page`<br>25.6s · 24978 tokens · Phoenix trace `b09746c2be8ad5640e0ee88defefa26f`</sub>

## 9. Traces in Phoenix

Open http://localhost:6006 and go to project **datasheet-agent**. Each question is one trace:

```
answer_question                       CHAIN   (question → final structured answer; verdict, revisions, tool-call count)
├─ invoke_agent datasheet_agent       AGENT
│   ├─ chat gpt-5.5                   LLM     (reasoning step: messages, tool calls, tokens)
│   ├─ execute_tool search_datasheets TOOL
│   │    └─ search_datasheets         RETRIEVER (query + ranked documents)
│   ├─ execute_tool view_page_image   TOOL
│   └─ chat gpt-5.5 …                 LLM     (final structured answer)
├─ invoke_agent verifier              AGENT → LLM
└─ (revision round + re-verification, if needed)
```

Look up a specific answer with the `trace_id` printed under it.